# 真实集成：SelfCheck NLI

真实 API 采样段落，本地 NLI 按官方顺序将原句作为 premise、采样段落作为 hypothesis，输出实际矛盾概率。

## 真实集成前提

安装：`python3 -m pip install -e ".[openai,nli]"`；需要 API key 和本地 NLI 模型。

代码单元没有预执行。填写 API 凭据或本地模型绝对路径后，从上到下运行；不要将 API key 保存到 notebook。

## 参数说明

- `LOCAL_NLI_MODEL: str`：本地 NLI 模型目录。
- `NUM_SAMPLES: int = 5`：每句独立样本数。
- `TEMPERATURE: float = 0.8`：采样多样性。

## 步骤 1：配置真实 API 与 NLI

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if not (repo_root / "src" / "hallucination").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import os
from getpass import getpass
from openai import OpenAI
from hallucination.detection.backends.api import OpenAIResponsesBackend
OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0.8
api_key = os.environ.get("OPENAI_API_KEY") or getpass("OpenAI API key: ")
base_url = os.environ.get("OPENAI_BASE_URL")
client = OpenAI(api_key=api_key, **({"base_url": base_url} if base_url else {}))
backend = OpenAIResponsesBackend(OPENAI_MODEL, client=client, temperature=TEMPERATURE)

from hallucination.detection.backends.nli import BidirectionalNLIBackend
LOCAL_NLI_MODEL = "/absolute/path/to/your/nli-model"
NLI_DEVICE = "cpu"
if not Path(LOCAL_NLI_MODEL).is_dir():
    raise ValueError("Set LOCAL_NLI_MODEL to an existing local NLI model directory.")
nli = BidirectionalNLIBackend(LOCAL_NLI_MODEL, device=NLI_DEVICE)

## 步骤 2：运行 SelfCheck NLI

In [ ]:
from hallucination.detection.blackbox.selfcheck_nli import SelfCheckNLIScorer
from hallucination.detection.pipelines.short_answer import BlackBoxSamplingPipeline
result = BlackBoxSamplingPipeline(backend, sentence_scorers=[SelfCheckNLIScorer(nli)], num_samples=5).detect_prompt("请用两句话介绍蓝鲸的食性和分类。")
for unit in result.units:
    print(unit.text, unit.method_scores[0].risk, unit.method_scores[0].raw["comparisons"])

## 结果解读与限制

NLI 输出是分类模型置信度而非事实真值；模型重复同一错误时，自一致性风险依然可能很低。